# GrandNexus v7.3.0-rc — check on Colab (logbook 64-65)
Point 4 of the work order: `boot(profile="minimal")`, NexusCore as orchestrator only, every module tagged (`MODULES.md`).
Measured offline before delivery: minimal profile 17 modules, boot 0.05 s, same answers as the full profile on 31 items.
This notebook replays that on Colab (**0 API call**) and confirms the prompt cache of v7.2.5-rc on the real API
(**about 6 calls**, 5 FOLIO validation items already seen, a few cents). If everything passes, cell 6 tags **v7.3.0**.

Expected: cell 4 prints four PASS lines; cell 5 prints `the cache works` and a `cost TOTAL ... saved N%` line; cell 6 prints `tagged v7.3.0`.
Run the cells in order. **If a cell prints STOP, do not run the next ones: send the output.** If the Colab assistant offers to
rewrite a cell, decline. No item text is printed. Send back the output of cells 4, 5 and 6, and the zip named in cell 6.

## 1. Get exactly v7.3.0-rc

In [ ]:
# 1. Get EXACTLY GrandNexus v7.3.0-rc (never another zip; stop if it is not found)
import os, glob, fnmatch, subprocess, zipfile, shutil, time
VERSION, ZIP_PATTERN = 'v7.3.0-rc', 'GrandNexus_v7.3.0-rc*.zip'
CONTENT, WORK = '/content', '/content/GrandNexus'
MYDRIVE = '/content/drive/MyDrive'

def gn_version(path):
    """(git describe, has the story-aware bench) of a GrandNexus folder, or (None, False)."""
    if not os.path.isdir(os.path.join(path, '.git')):
        return None, False
    d = subprocess.run('git describe --tags', shell=True, cwd=path, capture_output=True, text=True).stdout.strip()
    bench = os.path.join(path, 'tools', 'bench_learning.py')
    ok = all(os.path.exists(os.path.join(path, 'tools', f)) for f in ('colab_save.py', 'cost.py', 'boot_profiles.py')) and \
         os.path.exists(bench) and '--story-mode' in open(bench, encoding='utf-8').read()
    return d, ok

try:
    from google.colab import drive; drive.mount(os.path.join(CONTENT, 'drive'))
except Exception as e:
    print('Drive not mounted:', e)

# the release zip, newest first: My Drive/GrandNexus and My Drive first, then anywhere in My Drive (3 levels deep);
# any other GrandNexus zip (older versions, session saves) is ignored
found = glob.glob(os.path.join(MYDRIVE, 'GrandNexus', ZIP_PATTERN)) + glob.glob(os.path.join(MYDRIVE, ZIP_PATTERN))
if not found:
    for root, dirs, names in os.walk(MYDRIVE):
        if root[len(MYDRIVE):].count(os.sep) >= 3:
            dirs[:] = []
        found += [os.path.join(root, n) for n in names if fnmatch.fnmatch(n, ZIP_PATTERN)]
found.sort(key=os.path.getmtime)
if found:
    source = found[-1]
else:
    print(f'{ZIP_PATTERN} not found in My Drive: upload it now (any other zip is refused).')
    try:
        from google.colab import files
        up = next(iter(files.upload()))
    except Exception as e:
        raise SystemExit(f'STOP: no {ZIP_PATTERN} on the Drive and no upload possible ({e}).')
    if not fnmatch.fnmatch(os.path.basename(up), ZIP_PATTERN):
        raise SystemExit(f'STOP: {up} is not {ZIP_PATTERN}.')
    source = os.path.join(CONTENT, up)
print('release zip:', source)

# an existing /content/GrandNexus of another version is moved aside, never reused, never deleted
d, ok = gn_version(WORK)
if os.path.exists(WORK) and not (d or '').startswith(VERSION):
    aside = WORK + '_aside_' + time.strftime('%Y%m%d_%H%M%S')
    shutil.move(WORK, aside); print(f'existing {WORK} ({d}) moved aside to {aside}')
if not os.path.exists(WORK):
    z = zipfile.ZipFile(source)
    top = {n.split('/')[0] for n in z.namelist() if n.strip('/')}
    z.extractall(CONTENT if top == {'GrandNexus'} else WORK)
    print('extracted')
subprocess.run('git checkout -q main', shell=True, cwd=WORK)
d, ok = gn_version(WORK)
print('git describe:', d, '| v7.3.0-rc tools present:', ok)
print(subprocess.run('git log --oneline -1 && git branch', shell=True, cwd=WORK, capture_output=True, text=True).stdout)
if not ((d or '').startswith(VERSION) and ok):
    raise SystemExit(f'STOP: {WORK} is {d}, not {VERSION}. Do not run the next cells.')
print(f'OK: {VERSION}')


## 2. Packages, key, pre-flight (1 call)

In [ ]:
# 2. Packages, key, pre-flight check (1 call, through the translator itself: the cached system prompt must be accepted)
import subprocess, sys, os
WORK = '/content/GrandNexus'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anthropic', 'clingo', 'sympy', 'networkx', 'psutil'])
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
sys.path.insert(0, WORK)
from grandnexus.llm_interface.translator_v5 import Translator, SYSTEM_TRANSLATE
tr = Translator(model='claude-sonnet-5-5')
reply = tr._call(SYSTEM_TRANSLATE, 'TEXT:\nAll dogs bark. Rex is a dog.\n\nQUESTION (or statement to verify):\nDoes Rex bark?', 'preflight')
r = tr.receipts[-1]
print('reply received:', bool(reply.strip()), '| id:', r.id, '| model:', r.model)
print(f'usage: input {r.tokens_in} | output {r.tokens_out} | cache write {r.cache_write} | cache read {r.cache_read}')
print('the system prompt went into the cache' if (r.cache_write or r.cache_read) else 'WARNING: no cache write or read on this call (send this output)')


## 3. Offline tests (0 calls)

In [ ]:
# 3. Offline tests (0 calls)
import subprocess, sys, glob, os
WORK = '/content/GrandNexus'
bad = []
for t in sorted(glob.glob(os.path.join(WORK, 'tests_lab', 'test_*.py'))):
    r = subprocess.run([sys.executable, '-W', 'ignore', t], cwd=WORK, capture_output=True, text=True)
    last = (r.stdout.strip().splitlines() or ['(no output)'])[-1]
    ok = r.returncode == 0 and not last.startswith('FAIL') and 'Traceback' not in r.stderr
    print(f"{'ok ' if ok else 'BAD'} {os.path.basename(t):40} {last[:70]}")
    if not ok:
        bad.append(t)
if bad:
    raise SystemExit(f'STOP: {len(bad)} offline test(s) failed; send this output.')


## 4. Boot profiles (0 calls)

In [ ]:
# 4. Boot profiles (0 calls): full vs minimal, laboratory and live mode, criteria of logbook 64
import subprocess, sys, glob, json, os
WORK = '/content/GrandNexus'
p = subprocess.Popen([sys.executable, '-W', 'ignore', 'tools/boot_profiles.py'], cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    sys.stdout.write(line)
p.wait()
res = sorted(glob.glob(os.path.join(WORK, 'tests_lab', 'resultats', 'boot', 'boot_profiles_*.json')), key=os.path.getmtime)
if not res:
    raise SystemExit('STOP: no result written; send this output.')
crit = json.load(open(res[-1]))['criteria']
print('\nall criteria pass' if all(c['pass'] for c in crit.values()) else '\nNOT ALL CRITERIA PASS (send this output)')


## 5. Prompt cache on the real API (about 6 calls)

In [ ]:
# 5. Cost check (about 6 calls): 5 validation items already seen (logbook 40), translator, strict verdict
import subprocess, sys, glob, json, os
WORK = '/content/GrandNexus'
CMD = 'python tools/bench_folio.py --split validation --n 5 --seed 0 --modes translator --strict'
p = subprocess.Popen(CMD, shell=True, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    sys.stdout.write(line)
p.wait()
res = sorted(glob.glob(os.path.join(WORK, 'tests_lab', 'resultats', 'folio_validation_5_*.json')), key=os.path.getmtime)
if not res:
    raise SystemExit('STOP: no result written; send this output.')
rows = json.load(open(res[-1]))['rows']['translator']
reads = sum(r.get('cache_read', 0) for r in rows); writes = sum(r.get('cache_write', 0) for r in rows)
print(f'\ncache: {writes} tokens written, {reads} tokens read over {sum(r.get("calls", 0) for r in rows)} calls ->',
      'the cache works' if reads > 0 else 'NO CACHE READ (send this output)')


## 6. Release decision and save

In [ ]:
# 6. Release decision and save: tag v7.3.0 only if the profiles pass AND the prompt cache works on the real API
import subprocess, sys, glob, json, os
WORK = '/content/GrandNexus'
boot = sorted(glob.glob(WORK + '/tests_lab/resultats/boot/boot_profiles_*.json'), key=os.path.getmtime)
cost = sorted(glob.glob(WORK + '/tests_lab/resultats/folio_validation_5_*.json'), key=os.path.getmtime)
ok_profiles = bool(boot) and all(c['pass'] for c in json.load(open(boot[-1]))['criteria'].values())
reads = sum(r.get('cache_read', 0) for r in json.load(open(cost[-1]))['rows']['translator']) if cost else 0
release = ok_profiles and reads > 0
print('profiles:', 'PASS' if ok_profiles else 'FAIL or missing', '| prompt cache read on the real API:', reads, 'tokens')
git = lambda *a: subprocess.run(['git', *a], cwd=WORK, capture_output=True, text=True).stdout.strip()
git('add', '-A')
git('-c', 'user.name=Khaled', '-c', 'user.email=khaled@grandnexus.local', 'commit', '-qm',
    'v7.3.0 check on Colab: ' + ('all criteria pass' if release else 'NOT all criteria pass'))
if release:
    git('tag', 'v7.3.0')
    print('tagged v7.3.0')
else:
    print('no release tag: send this output')
r = subprocess.run([sys.executable, 'tools/colab_save.py', '--tag', 'check', '--expect',
                    'tests_lab/resultats/boot/boot_profiles_*.json', 'tests_lab/resultats/folio_validation_5_*.json'],
                   cwd=WORK, capture_output=True, text=True)
print(r.stdout, r.stderr[-500:])
